# Food Rescue CV: Data Pipeline Setup
Food-101 loading, preprocessing, and stratified data-fraction splits (10%/50%/100%) for ResNet18 vs ViT-B/16 comparison.

## 1. Mount Google Drive
This keeps data, checkpoints, and logs persistent across Colab session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/food-rescue-cv'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
print('Project dir ready at', PROJECT_DIR)

## 2. Install packages

In [ ]:
!pip install -q timm
%pip install -Uq "mlop[full]"

import torch
import torchvision
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 3. Load Food-101
Downloads once to local Colab disk (faster than Drive, not persisted across sessions, only takes ~5 min to redownload). `torchvision.datasets.Food101` handles train/test splits automatically.

In [ ]:
from torchvision.datasets import Food101

DATA_ROOT = '/content/food101_data'  # local disk, faster download, not persisted

train_set = Food101(root=DATA_ROOT, split='train', download=True)
test_set = Food101(root=DATA_ROOT, split='test', download=True)

print('Train size:', len(train_set))
print('Test size:', len(test_set))
print('Num classes:', len(train_set.classes))
print('Sample classes:', train_set.classes[:5])

In [ ]:
# Spot check: view a sample image and its label
import matplotlib.pyplot as plt

img, label = train_set[0]
plt.imshow(img)
plt.title(f'Label: {train_set.classes[label]}')
plt.axis('off')
plt.show()

## 4. Preprocessing
Shared transforms for ResNet18 and ViT-B/16 (both pretrained on ImageNet). Keeping preprocessing identical across models so performance differences come from architecture, not input handling.

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Reload datasets with transforms applied
train_set = Food101(root=DATA_ROOT, split='train', transform=train_transform, download=False)
test_set = Food101(root=DATA_ROOT, split='test', transform=eval_transform, download=False)

## 5. Stratified data-fraction splits (10% / 50% / 100%)
Each class proportionally represented at every fraction, reproducible via fixed seed.

In [ ]:
import numpy as np
import random
from collections import defaultdict
from torch.utils.data import Subset

SEED = 42

def stratified_subset(dataset, fraction, seed=SEED):
    """Return a Subset containing `fraction` of each class, stratified."""
    if fraction >= 1.0:
        return dataset

    rng = np.random.default_rng(seed)

    # Group indices by class label
    class_indices = defaultdict(list)
    for idx, label in enumerate(dataset._labels):
        class_indices[label].append(idx)

    selected = []
    for label, indices in class_indices.items():
        indices = np.array(indices)
        rng.shuffle(indices)
        n_keep = max(1, int(len(indices) * fraction))
        selected.extend(indices[:n_keep].tolist())

    return Subset(dataset, selected)


# Build the three training splits
train_10 = stratified_subset(train_set, 0.10)
train_50 = stratified_subset(train_set, 0.50)
train_100 = train_set  # full dataset

print('10% split size:', len(train_10))
print('50% split size:', len(train_50))
print('100% split size:', len(train_100))

## 6. DataLoaders
Quick sanity check that batches load correctly before wiring up training.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_CLASSES = 101

def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader_100 = make_loader(train_100)
test_loader = make_loader(test_set, shuffle=False)

# Sanity check: pull one batch
images, labels = next(iter(train_loader_100))
print('Batch shape:', images.shape)
print('Labels shape:', labels.shape)

## 7. Shared setup: device, seeds, mlop
Everything below (ViT section and ResNet18 section) builds on this. Run once per session.

In [ ]:
import torch.nn as nn
import mlop

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# If mlop.ai is down (auth/session errors), set this to True to skip mlop logging
# entirely and just track results locally instead. Every mlop call elsewhere in this
# notebook is wrapped so a failed call won't crash a run, but flipping this avoids
# repeated failed requests slowing things down.
LOCAL_FALLBACK = False

if not LOCAL_FALLBACK:
    try:
        mlop.login()
    except Exception as e:
        print(f'mlop.login failed ({e}), continuing without live logging. Set LOCAL_FALLBACK = True to silence this.')

## 8. Shared model-agnostic train/eval functions
Used by both the ViT and ResNet18 sections below, so logs and metrics are directly comparable. Includes a validation split (Food-101 only ships train/test natively, so we hold out 10% of each training split) so overfitting can be tracked per epoch, not just judged from a single final test number.

In [ ]:
from torch.utils.data import random_split
from tqdm.auto import tqdm


def make_train_val_loaders(dataset, val_frac=0.10, batch_size=BATCH_SIZE, seed=SEED):
    n_val = int(len(dataset) * val_frac)
    n_train = len(dataset) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(dataset, [n_train, n_val], generator=generator)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader


def train_one_epoch(model, loader, optimizer, criterion, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(loader, desc=f'Epoch {epoch} [train]', leave=True)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress_bar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{correct/total:.4f}'})
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, epoch, split_name='val'):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(loader, desc=f'Epoch {epoch} [{split_name}]', leave=True)
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress_bar.set_postfix({'loss': f'{total_loss/total:.4f}', 'acc': f'{correct/total:.4f}'})
    return total_loss / total, correct / total

---
# Part A: ViT-B/16 (Arjot)

**Fixes applied to your original cells:**
1. **Pretrained weights bug**: `vit_b_16()` was loading random-initialized weights despite `config['pretrained'] = True`. Fixed to `vit_b_16(weights=ViT_B_16_Weights.DEFAULT)`. This is why the first test run only hit ~7.8% train accuracy after one epoch, that's consistent with training from scratch, not fine-tuning.
2. Added validation split + per-epoch val logging (reuses `make_train_val_loaders` above).
3. Added checkpoint saving to Drive (`best.pt` and `latest.pt` per run) so a Colab disconnect doesn't lose progress.
4. Wrapped as a reusable `run_vit_experiment` function, mirroring the ResNet18 section below, so both of you can run the same data-fraction sweep structure and merge results cleanly.

Run this section to redo the smoke test with the corrected pretrained weights, then use it as the basis for your own tuning + fraction sweep (same shape as Part B, just swap in ViT).

In [ ]:
from torchvision.models import vit_b_16, ViT_B_16_Weights

def run_vit_experiment(train_dataset, fraction_label, epochs=10, aug_label='aug_on',
                        lr=1e-4, wd=1e-4, phase='sweep'):
    run_name = f'vit_b16_frac{fraction_label}_{aug_label}' if phase == 'sweep' else f'vit_b16_tune_lr{lr}_wd{wd}'
    checkpoint_dir = f'{PROJECT_DIR}/checkpoints/{run_name}'
    os.makedirs(checkpoint_dir, exist_ok=True)

    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)  # FIX: was vit_b_16() with no weights
    input_features = model.heads.head.in_features
    model.heads.head = nn.Linear(input_features, NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    train_loader, val_loader = make_train_val_loaders(train_dataset)

    config = {
        'model': 'vit_b_16', 'pretrained': True, 'data_fraction': fraction_label,
        'augmentation': aug_label, 'learning_rate': lr, 'weight_decay': wd,
        'epochs': epochs, 'batch_size': BATCH_SIZE, 'seed': SEED, 'phase': phase,
    }

    op = None
    if not LOCAL_FALLBACK:
        try:
            op = mlop.init(name=run_name, project='food-rescue-cv', config=config)
        except Exception as e:
            print(f'mlop.init failed ({e}), continuing without live logging for this run')

    best_val_acc = 0.0
    test_acc = None

    try:
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, epoch)
            val_loss, val_acc = evaluate(model, val_loader, criterion, epoch, split_name='val')

            if op is not None:
                try:
                    op.log({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                             'val_loss': val_loss, 'val_acc': val_acc})
                except Exception as e:
                    print(f'mlop.log failed ({e}), continuing')

            print(f'Epoch {epoch:02d}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.4f} '
                  f'| val loss {val_loss:.4f} acc {val_acc:.4f}')

            if phase == 'sweep':
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(), 'val_acc': val_acc,
                            'config': config}, f'{checkpoint_dir}/latest.pt')
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                                'val_acc': val_acc, 'config': config}, f'{checkpoint_dir}/best.pt')
            else:
                best_val_acc = max(best_val_acc, val_acc)

        if phase == 'sweep':
            test_loss, test_acc = evaluate(model, test_loader, criterion, epochs, split_name='test')
            if op is not None:
                try:
                    op.log({'test_loss': test_loss, 'test_acc': test_acc})
                except Exception as e:
                    print(f'mlop.log failed ({e}), continuing')
            print(f'Final test | loss {test_loss:.4f} acc {test_acc:.4f}')

    finally:
        if op is not None:
            try:
                op.finish()
            except Exception:
                pass

    return model, {'best_val_acc': best_val_acc, 'test_acc': test_acc}

In [ ]:
# Smoke test with corrected pretrained weights, 1 epoch on 100% data
_ = run_vit_experiment(train_100, fraction_label='100', epochs=1)

---
# Part B: ResNet18 (Ash)

Hyperparameter tuning on the 10% split (same grid as Arjot: `lr in [5e-5, 1e-4, 5e-4, 1e-2]`, `weight_decay in [1e-4, 1e-2]`), then the full 10%/50%/100% sweep with the winning config.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

def run_resnet18_experiment(train_dataset, fraction_label, epochs=10, aug_label='aug_on',
                             lr=1e-4, wd=1e-4, phase='sweep'):
    run_name = f'resnet18_frac{fraction_label}_{aug_label}' if phase == 'sweep' else f'resnet18_tune_lr{lr}_wd{wd}'
    checkpoint_dir = f'{PROJECT_DIR}/checkpoints/{run_name}'
    os.makedirs(checkpoint_dir, exist_ok=True)

    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    train_loader, val_loader = make_train_val_loaders(train_dataset)

    config = {
        'model': 'resnet18', 'pretrained': True, 'data_fraction': fraction_label,
        'augmentation': aug_label, 'learning_rate': lr, 'weight_decay': wd,
        'epochs': epochs, 'batch_size': BATCH_SIZE, 'seed': SEED, 'phase': phase,
    }

    op = None
    if not LOCAL_FALLBACK:
        try:
            op = mlop.init(name=run_name, project='food-rescue-cv', config=config)
        except Exception as e:
            print(f'mlop.init failed ({e}), continuing without live logging for this run')

    best_val_acc = 0.0
    test_acc = None

    try:
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, epoch)
            val_loss, val_acc = evaluate(model, val_loader, criterion, epoch, split_name='val')

            if op is not None:
                try:
                    op.log({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                             'val_loss': val_loss, 'val_acc': val_acc})
                except Exception as e:
                    print(f'mlop.log failed ({e}), continuing')

            print(f'Epoch {epoch:02d}/{epochs} | train loss {train_loss:.4f} acc {train_acc:.4f} '
                  f'| val loss {val_loss:.4f} acc {val_acc:.4f}')

            if phase == 'sweep':
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(), 'val_acc': val_acc,
                            'config': config}, f'{checkpoint_dir}/latest.pt')
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                                'val_acc': val_acc, 'config': config}, f'{checkpoint_dir}/best.pt')
            else:
                best_val_acc = max(best_val_acc, val_acc)

        if phase == 'sweep':
            test_loss, test_acc = evaluate(model, test_loader, criterion, epochs, split_name='test')
            if op is not None:
                try:
                    op.log({'test_loss': test_loss, 'test_acc': test_acc})
                except Exception as e:
                    print(f'mlop.log failed ({e}), continuing')
            print(f'Final test | loss {test_loss:.4f} acc {test_acc:.4f}')

    finally:
        if op is not None:
            try:
                op.finish()
            except Exception:
                pass

    return model, {'best_val_acc': best_val_acc, 'test_acc': test_acc}

## B1. Hyperparameter tuning (10% data, short runs)

In [ ]:
learning_rates = [5e-5, 1e-4, 5e-4, 1e-2]
weight_decays = [1e-4, 1e-2]

tuning_results = []
for lr in learning_rates:
    for wd in weight_decays:
        print(f'--- Tuning lr={lr}, wd={wd} ---')
        _, metrics = run_resnet18_experiment(train_10, fraction_label='10', epochs=3, lr=lr, wd=wd, phase='tuning')
        tuning_results.append({'lr': lr, 'weight_decay': wd, 'best_val_acc': metrics['best_val_acc']})

tuning_results.sort(key=lambda r: r['best_val_acc'], reverse=True)
print('Best config:', tuning_results[0])

## B2. Full sweep: best config across 10% / 50% / 100% data

In [ ]:
BEST_LR = tuning_results[0]['lr']
BEST_WD = tuning_results[0]['weight_decay']
print(f'Using best config: lr={BEST_LR}, weight_decay={BEST_WD}')

# Smoke test on 10% data, 1 epoch, before committing to the full 10-epoch runs
_ = run_resnet18_experiment(train_10, fraction_label='10', epochs=1, lr=BEST_LR, wd=BEST_WD)

In [ ]:
EPOCHS = 10

sweep_results = {}
for fraction_label, dataset in [('10', train_10), ('50', train_50), ('100', train_100)]:
    print(f'--- Running ResNet18 on {fraction_label}% data ---')
    model, metrics = run_resnet18_experiment(dataset, fraction_label=fraction_label, epochs=EPOCHS, lr=BEST_LR, wd=BEST_WD)
    sweep_results[fraction_label] = metrics

print(sweep_results)

## Save results locally (backup in case mlop.ai is unavailable)

In [ ]:
import json
with open(f'{PROJECT_DIR}/results/resnet18_results.json', 'w') as f:
    json.dump({'tuning': tuning_results, 'sweep': sweep_results}, f, indent=2)
print('Saved to', f'{PROJECT_DIR}/results/resnet18_results.json')

## Next steps
- Arjot: run the same tuning (B1-style) + sweep (B2-style) structure for ViT-B/16 using `run_vit_experiment`, save to `{PROJECT_DIR}/results/vit_results.json`
- Both: merge `resnet18_results.json` and `vit_results.json` into one comparison table/plots
- Both: pick the single best-performing model/fraction combo, rerun with `aug_label='aug_off'` (disable `RandomHorizontalFlip` in the train transform) for the augmentation ablation